In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    roc_auc_score
)
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
warnings.filterwarnings("ignore")

In [ ]:
# Cargamos de datos
df = pd.read_excel("Data/datos_limpios.xlsx")
df.head()

ANALISIS EXPLORATIORIO

In [ ]:

df.info()

In [ ]:
#ESTADISTICAS DESCRIPTIVAS
display(df.describe())

In [ ]:
#VEMOS PROPORCION DE LA VARIABLE OBJETIVO
print("Número de observaciones por clase:")
print(df["Impago"].value_counts())
print("\nPorcentaje de observaciones por clase:")
print(df["Impago"].value_counts(normalize=True) * 100)

In [ ]:
df.columns

In [ ]:
# Variables predictoras (X) y objetivo (y)
X = df[['Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes',
       'Ratio_Deuda_Ingresos', 'Estudios', 'Fiador']] 
y = df["Impago"]

# División Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.8,
    random_state=42,
    shuffle=True,
    stratify=y
)

Estandarizamos el dataset ya que los datos son muy desproporcionados

In [ ]:
scaler = StandardScaler()
# Ajustamos y transformamos Train, solo transformamos Test
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

Realizamos undersampling y oversampling para igualar la cantidad de instancias que hay de la variable objetivo ya que esta muy desproporcionada

In [ ]:
# # 1. Undersampling: Reducimos la clase mayoritaria 0
# undersampler = RandomUnderSampler(sampling_strategy={0: 30000}, random_state=42)
# X_train_under, y_train_under = undersampler.fit_resample(X_train_scaled, y_train)

# # 2. SMOTE: Aumentamos la clase minoritaria 1
# smote = SMOTE(sampling_strategy={1: 30000}, random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train_under, y_train_under)

# print(f"Distribución final para entrenar -> 0: {sum(y_train_resampled==0)} | 1: {sum(y_train_resampled==1)}")

In [ ]:
# Definimos los modelos base y los hiperparámetros que probará el Random Search
configuracion_modelos = {
    "Ridge (L2)": {
        "modelo": LogisticRegression(penalty="l2", solver="saga", max_iter=2000, random_state=42),
        "parametros": {
            "C": np.logspace(-4, 4, 50)
        }
    },
    "Lasso (L1)": {
        "modelo": LogisticRegression(penalty="l1", solver="saga", max_iter=2000, random_state=42),
        "parametros": {
            "C": np.logspace(-4, 4, 50)
        }
    },
    "ElasticNet (L1 + L2)": {
        "modelo": LogisticRegression(penalty="elasticnet", solver="saga", max_iter=2000, random_state=42),
        "parametros": {
            "C": np.logspace(-4, 4, 50),
            "l1_ratio": np.linspace(0.1, 0.9, 9)
        }
    }
}

# Listas y variables para guardar los resultados de cada iteración
resultados_logistica = []
mejor_auc_global = 0.0
mejor_modelo_log = None
mejor_nombre_log = ""

In [ ]:
for nombre_modelo, config in configuracion_modelos.items():
    print("=====================================================================")
    print(f" ENTRENANDO Y EVALUANDO MODELO: {nombre_modelo}")
    print("=====================================================================")
    
    # 1. Configuramos el Random Search con Validación Cruzada (cv=5)
    random_search = RandomizedSearchCV(
        estimator=config["modelo"],
        param_distributions=config["parametros"],
        n_iter=15,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        random_state=42
    )
    
    # 2. Entrenamos con los datos ORIGINALES ESTANDARIZADOS (Sin SMOTE)
    random_search.fit(X_train_scaled, y_train)
    
    # Extraemos el modelo ganador de esta regularización
    modelo_actual = random_search.best_estimator_
    mejores_params = random_search.best_params_
    print(f"-> Mejores hiperparámetros encontrados (CV-5): {mejores_params}\n")
    
    # 3. Tabla de importancia de variables
    coeficientes = np.atleast_1d(modelo_actual.coef_[0])
    resumen_coeficientes = pd.DataFrame({
        "Variable": X_train.columns,
        "Coeficiente": coeficientes,
        "Importancia_Absoluta": np.abs(coeficientes)
    }).sort_values(by="Importancia_Absoluta", ascending=False).reset_index(drop=True)
    
    print("--- IMPORTANCIA DE VARIABLES ---")
    print(resumen_coeficientes.to_string(index=False))
    
    # 4. Cálculo de probabilidades
    prob_train = modelo_actual.predict_proba(X_train_scaled)[:, 1]
    roc_train = roc_auc_score(y_train, prob_train)
    
    probabilidades_clase_1 = modelo_actual.predict_proba(X_test_scaled)[:, 1]
    auc_score = roc_auc_score(y_test, probabilidades_clase_1)
    
    # 5. CÁLCULO DEL UMBRAL ÓPTIMO PARA MAXIMIZAR F1-SCORE
    prec, rec, th_pr = precision_recall_curve(y_test, probabilidades_clase_1)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-10)
    umbral_optimo = th_pr[np.argmax(f1_scores[:-1])]
    
    clasificacion_test = np.where(probabilidades_clase_1 >= umbral_optimo, 1, 0)
    
    # 6. Métricas de error
    acierto = accuracy_score(y_test, clasificacion_test)
    error = 1.0 - acierto
    
    print(f"\n--- MÉTRICAS DE ERROR ({nombre_modelo}) ---")
    print(f"- ROC-AUC en Entrenamiento: {roc_train:.4f}")
    print(f"- ROC-AUC en Test:          {auc_score:.4f}")
    print(f"- Umbral matemático óptimo (F1): {umbral_optimo:.4f}")
    print(f"- Accuracy (Acierto):       {acierto * 100:.2f}%")
    print(f"- Error Global:             {error * 100:.2f}%\n")
    print(classification_report(y_test, clasificacion_test))
    
    # 7. Panel de 3 gráficos
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Diagnóstico Gráfico: {nombre_modelo}", fontsize=14, fontweight="bold")
    
    c_matrix = confusion_matrix(y_test, clasificacion_test)
    sns.heatmap(c_matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_title("- Matriz de Confusión")
    axes[0].set_xlabel("Predicción")
    axes[0].set_ylabel("Real")
    
    axes[1].plot(rec, prec, color="purple", lw=2)
    axes[1].set_xlabel("Recall (Sensibilidad)")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("- Curva PR")
    axes[1].grid(True, alpha=0.3)
    
    fpr, tpr, thresholds_roc = roc_curve(y_test, probabilidades_clase_1)
    axes[2].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {auc_score:.2f}")
    axes[2].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
    axes[2].set_xlabel("FPR (Falsos Positivos)")
    axes[2].set_ylabel("TPR (Verdaderos Positivos)")
    axes[2].set_title("- Curva ROC-AUC")
    axes[2].legend(loc="lower right")
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Guardamos los resultados
    resultados_logistica.append({
        "Modelo": nombre_modelo,
        "Mejores Hiperparametros": str(mejores_params),
        "ROC_AUC Train": roc_train,
        "ROC_AUC Test": auc_score,
        "Accuracy Test (%)": round(acierto * 100, 2),
        "Error Test (%)": round(error * 100, 2)
    })
    
    if auc_score > mejor_auc_global:
        mejor_auc_global = auc_score
        mejor_modelo_log = modelo_actual
        mejor_nombre_log = nombre_modelo

ya que tenemos el modelo, ahora vamos a crear una nueva columna que diga la propabilidad de impago de cada cliente

In [ ]:
# Mostramos la tabla resumen
df_comparativa_log = pd.DataFrame(resultados_logistica)
print(f"MODELO GANADOR SELECCIONADO: {mejor_nombre_log} (ROC-AUC Test: {mejor_auc_global:.4f})")
display(df_comparativa_log)

# Estandarizamos todo el conjunto de datos originales y generamos la probabilidad
X_total_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)
probabilidades_totales = mejor_modelo_log.predict_proba(X_total_scaled)

df["prob_impago"] = probabilidades_totales[:, 1]

print("\n--- PROBABILIDADES DE IMPAGO POR CLIENTE (PRIMERAS 10 FILAS) ---")
print(df[["ID", "Impago", "prob_impago"]].head(10))

In [ ]:
#dame el codigo para poder ver los maximos, minimos y promedio de la probabilidad de impago en base a la columna de impago
impago_stats = df.groupby("Impago")["prob_impago"].agg(['min', 'max', 'mean', 'std',"median"]).reset_index()
print(impago_stats)

In [ ]:
df.to_excel("Data/dataset_con_probabilidades_2.xlsx", index=False)